# Analyse
- import des incendies et des communes en .parquet
- jointure
- analyse

In [ ]:
import sys
from pathlib import Path

import chardet
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster
import matplotlib.pyplot as plt
import seaborn as sns

# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW_DIR, DATA_CLEAN_DIR, GEO_DATA_CLEAN_DIR
from utils.cleaning_utils import delete, normalize_columns_names, normalize_text_columns_cells, optimize_numeric_column, fill_rate
from utils.analysis_utils import plot_missing_bar, plot_numeric_histograms, plot_corr_heatmap, plot_qualitative

In [13]:
df_incendies = pd.read_parquet(
    DATA_CLEAN_DIR / "incendies.parquet"
)

df_communes = pd.read_parquet(
    GEO_DATA_CLEAN_DIR / "communes.parquet"
)

In [9]:
df_incendies.head(2)

,departement,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,surfaces_non_boisees_naturelles_m2,surfaces_non_boisees_artificialisees_m2,surfaces_non_boisees_m2,type_de_peuplement,nature
0,2a,2a198,2011-01-01 00:33:00,50,24,24,0,<NA>,<NA>,<NA>,<NA>,<NA>,2,1,involontaire (particulier)
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,4,NaN


In [14]:
df_communes.head(2)

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale
0,01001,1400,L'Abergement-Clémenciat,84,01,832,1565,53.0,-14,-50,16
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,-29,34,-20


#### Fusion des dataset incendies/BDIFF et liste des communes/data.gouv.fr
sur la clé code_insee

In [15]:
df = df_incendies.merge(
    df_communes,
    on="code_insee",
    how="left"
)

In [16]:
df.head(2)

,departement_x,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,...,localisation,nom_standard,region,departement_y,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale
0,2a,2a198,2011-01-01 00:33:00,50,24,24,0,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,11200,Lézignan-Corbières,76,11,10952,3815,287.0,58,19,-69


In [18]:
df_altitude_negative = df[df["altitude_maximale"] < 0]
df_altitude_negative

,departement_x,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,...,localisation,nom_standard,region,departement_y,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,11200,Lézignan-Corbières,76,11,10952,3815,287.0,58,19,-69
2,974,97415,2011-01-03 12:00:00,40000,40000,<NA>,0,<NA>,<NA>,0,...,97411,Saint-Paul,4,974,105240,24050,438.0,23,0,-117
3,11,11203,2011-01-06 11:01:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,11200,Lézignan-Corbières,76,11,10952,3815,287.0,58,19,-69
7,24,24037,2011-01-17 23:55:00,20,20,<NA>,0,<NA>,<NA>,0,...,24100,Bergerac,75,24,26323,5658,465.0,50,12,-110
8,07,07029,2011-01-18 12:26:00,5000,0,5000,<NA>,<NA>,<NA>,<NA>,...,7110,Beaumont,84,07,258,1929,13.0,14,-49,-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52795,976,97603,2025-12-04 09:04:00,21700,21000,<NA>,0,700,0,<NA>,...,97660,Bandrele,6,976,10282,3514,293.0,101,0,-128
52796,09,09325,2025-12-11 22:20:00,340000,115800,<NA>,224200,0,0,<NA>,...,9110,Vaychis,76,09,33,458,7.0,114,-22,-31
52799,09,09087,2025-12-18 14:38:00,820000,140000,<NA>,680000,0,0,<NA>,...,9250,Caussou,76,09,44,1582,3.0,-8,-8,-128
52802,47,47228,2025-12-27 16:46:46,3,3,<NA>,0,0,0,<NA>,...,47340,Saint-Antoine-de-Ficalba,75,47,715,1104,65.0,-65,108,-25


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52806 entries, 0 to 52805
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   departement_x                             52806 non-null  str    
 1   code_insee                                52806 non-null  object 
 2   date_de_premiere_alerte                   52806 non-null  str    
 3   surface_parcourue_m2                      52806 non-null  Int32  
 4   surface_foret_m2                          48156 non-null  Int32  
 5   surface_maquis_garrigues_m2               28073 non-null  Int32  
 6   autres_surfaces_naturelles_hors_foret_m2  34531 non-null  Int32  
 7   surfaces_agricoles_m2                     6588 non-null   Int32  
 8   autres_surfaces_m2                        6572 non-null   Int32  
 9   surface_autres_terres_boisees_m2          17028 non-null  Int32  
 10  surfaces_non_boisees_naturelles_m2        106

In [12]:
df.head()

,departement_x,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,...,localisation,nom_standard,region,departement_y,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale
0,2a,2a198,2011-01-01 00:33:00,50,24,24,0,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,11200,Lézignan-Corbières,76,11,10952,3815,287.0,58,19,-69
2,974,97415,2011-01-03 12:00:00,40000,40000,<NA>,0,<NA>,<NA>,0,...,97411,Saint-Paul,4,974,105240,24050,438.0,23,0,-117
3,11,11203,2011-01-06 11:01:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,11200,Lézignan-Corbières,76,11,10952,3815,287.0,58,19,-69
4,40,40113,2011-01-08 08:48:00,2000,2000,<NA>,0,<NA>,<NA>,0,...,40180,Goos,75,40,530,1061,50.0,40,4,73


typer les dates